In [0]:
raw_root = "abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/"

accounts_df = spark.read.json(f"{raw_root}accounts/")
deals_df = spark.read.json(f"{raw_root}deals/")
touches_df = spark.read.json(f"{raw_root}touches/")
history_df = spark.read.json(f"{raw_root}deal-history/")

print("Accounts:", accounts_df.count())
print("Deals:", deals_df.count())
print("Touches:", touches_df.count())
print("Deal history:", history_df.count())

Accounts: 1
Deals: 1
Touches: 1
Deal history: 1


In [0]:
from pyspark.sql.functions import explode, col, current_timestamp, lit

# ============================================================
# SALES CRM — BRONZE INGESTION
# ============================================================

raw_root = "abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/"

entities = [
    "accounts",
    "deals",
    "touches",
    "deal-history"
]

bronze = {}

for entity in entities:

    print(f"\n{'='*60}")
    print(f"Processing: {entity}")
    print(f"{'='*60}")

    # Read every dated load produced by ADF
    raw_df = (
        spark.read
        .option("multiline", "true")
        .json(f"{raw_root}{entity}/*/data.json")
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_entity", lit(entity))
    )

    # Flatten API envelope: data[] -> individual records
    records_df = (
        raw_df
        .select(
            explode(col("data")).alias("record"),
            "_source_file",
            "_ingested_at",
            "_entity"
        )
        .select(
            "record.*",
            "_source_file",
            "_ingested_at",
            "_entity"
        )
    )

    bronze[entity] = records_df

    print(f"Records: {records_df.count()}")

    display(records_df.limit(5))

print("\nBRONZE INGESTION COMPLETE")


Processing: accounts
Records: 100


account_id,account_name,country,created_date,industry,sales_rep_id,updated_at,_source_file,_ingested_at,_entity
1,Account 1,Saudi Arabia,2026-01-04,Healthcare,5,2026-07-20T08:00:00+00:00,abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/accounts/load_date=2026-10-01T12-28-17Z/data.json,2026-10-01T12:57:02.874Z,accounts
2,Account 2,United States,2025-06-03,Technology,2,2026-02-09T08:00:00+00:00,abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/accounts/load_date=2026-10-01T12-28-17Z/data.json,2026-10-01T12:57:02.874Z,accounts
3,Account 3,UAE,2025-02-08,Logistics,2,2026-08-25T08:00:00+00:00,abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/accounts/load_date=2026-10-01T12-28-17Z/data.json,2026-10-01T12:57:02.874Z,accounts
4,Account 4,UAE,2025-12-15,Retail,2,2026-05-18T08:00:00+00:00,abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/accounts/load_date=2026-10-01T12-28-17Z/data.json,2026-10-01T12:57:02.874Z,accounts
5,Account 5,United Kingdom,2025-07-28,Finance,3,2026-01-14T08:00:00+00:00,abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/accounts/load_date=2026-10-01T12-28-17Z/data.json,2026-10-01T12:57:02.874Z,accounts



Processing: deals
Records: 100


account_id,close_date,deal_id,deal_name,deal_value,open_date,stage,updated_at,_source_file,_ingested_at,_entity
1,2025-08-02,1,Deal 1 - Account 1,200518.91,2025-05-26,Won,2026-04-27T08:00:00+00:00,abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/deals/load_date=2026-10-01T12-44-39Z/data.json,2026-10-01T12:57:07.748Z,deals
1,null,2,Deal 2 - Account 1,106590.57,2025-01-10,Qualified,2026-06-12T08:00:00+00:00,abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/deals/load_date=2026-10-01T12-44-39Z/data.json,2026-10-01T12:57:07.748Z,deals
1,null,3,Deal 3 - Account 1,147206.57,2025-02-18,Proposal,2026-04-10T08:00:00+00:00,abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/deals/load_date=2026-10-01T12-44-39Z/data.json,2026-10-01T12:57:07.748Z,deals
2,2025-08-26,4,Deal 4 - Account 2,247202.11,2025-03-22,Lost,2026-09-23T08:00:00+00:00,abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/deals/load_date=2026-10-01T12-44-39Z/data.json,2026-10-01T12:57:07.748Z,deals
2,null,5,Deal 5 - Account 2,126513.31,2025-03-09,Qualified,2026-05-23T08:00:00+00:00,abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/deals/load_date=2026-10-01T12-44-39Z/data.json,2026-10-01T12:57:07.748Z,deals



Processing: touches
Records: 100


account_id,sales_rep_id,touch_date,touch_id,touch_type,touch_value,updated_at,_source_file,_ingested_at,_entity
1,3,2026-03-05,1,Email,3621.97,2026-08-10T08:00:00+00:00,abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/touches/load_date=2026-10-01T12-44-39Z/data.json,2026-10-01T12:57:12.603Z,touches
1,4,2026-03-08,2,Email,896.69,2026-04-01T08:00:00+00:00,abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/touches/load_date=2026-10-01T12-44-39Z/data.json,2026-10-01T12:57:12.603Z,touches
1,3,2026-02-27,3,Email,3788.93,2026-07-29T08:00:00+00:00,abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/touches/load_date=2026-10-01T12-44-39Z/data.json,2026-10-01T12:57:12.603Z,touches
1,5,2026-01-25,4,Demo,587.82,2026-03-30T08:00:00+00:00,abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/touches/load_date=2026-10-01T12-44-39Z/data.json,2026-10-01T12:57:12.603Z,touches
1,5,2026-04-20,5,Call,1111.52,2026-09-20T08:00:00+00:00,abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/touches/load_date=2026-10-01T12-44-39Z/data.json,2026-10-01T12:57:12.603Z,touches



Processing: deal-history
Records: 100


change_date,deal_id,history_id,new_stage,old_stage,updated_at,_source_file,_ingested_at,_entity
2026-07-16,1,1,Lead,Lead,2026-05-01T08:00:00+00:00,abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/deal-history/load_date=2026-10-01T12-44-39Z/data.json,2026-10-01T12:57:16.522Z,deal-history
2026-02-05,1,2,Proposal,Lead,2026-09-25T08:00:00+00:00,abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/deal-history/load_date=2026-10-01T12-44-39Z/data.json,2026-10-01T12:57:16.522Z,deal-history
2026-04-11,2,3,Qualified,Lead,2026-09-18T08:00:00+00:00,abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/deal-history/load_date=2026-10-01T12-44-39Z/data.json,2026-10-01T12:57:16.522Z,deal-history
2026-09-05,2,4,Qualified,Qualified,2026-04-18T08:00:00+00:00,abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/deal-history/load_date=2026-10-01T12-44-39Z/data.json,2026-10-01T12:57:16.522Z,deal-history
2026-05-04,2,5,Won,Qualified,2026-03-23T08:00:00+00:00,abfss://raw@shehabsalesdata235.dfs.core.windows.net/raw/deal-history/load_date=2026-10-01T12-44-39Z/data.json,2026-10-01T12:57:16.522Z,deal-history



BRONZE INGESTION COMPLETE


In [0]:
for entity, df in bronze.items():

    table_name = entity.replace("-", "_")

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"bronze_{table_name}")
    )

    print(f"✓ bronze_{table_name}")

✓ bronze_accounts
✓ bronze_deals
✓ bronze_touches
✓ bronze_deal_history


In [0]:
from pyspark.sql.functions import col, to_timestamp

silver = {}

for entity, df in bronze.items():

    clean = df

    # Convert obvious timestamp fields
    for field in ["updated_at", "created_at", "close_date", "touch_date"]:
        if field in clean.columns:
            clean = clean.withColumn(
                field,
                to_timestamp(col(field))
            )

    # Remove completely duplicated records
    clean = clean.dropDuplicates()

    silver[entity] = clean

    table_name = entity.replace("-", "_")

    (
        clean.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"silver_{table_name}")
    )

    print(f"✓ silver_{table_name}: {clean.count()} rows")

✓ silver_accounts: 100 rows
✓ silver_deals: 100 rows
✓ silver_touches: 100 rows
✓ silver_deal_history: 100 rows


In [0]:
from pyspark.sql.functions import (
    col,
    count,
    sum,
    when,
    coalesce,
    lit
)

# ------------------------------------------------------------
# Load Silver
# ------------------------------------------------------------

accounts = spark.table("silver_accounts")
deals = spark.table("silver_deals")
touches = spark.table("silver_touches")

# ------------------------------------------------------------
# Inspect stages once
# ------------------------------------------------------------

print("Deal stages:")
display(
    deals
    .select("stage")
    .distinct()
)

# ------------------------------------------------------------
# 1. Deals → account grain
# ------------------------------------------------------------

deal_summary = (
    deals
    .groupBy("account_id")
    .agg(
        count("*").alias("total_deals"),

        sum(
            when(col("stage") == "won", 1).otherwise(0)
        ).alias("won_deals"),

        sum(
            when(col("stage") == "lost", 1).otherwise(0)
        ).alias("lost_deals"),

        sum(
            when(
                col("stage") == "won",
                coalesce(col("deal_value"), lit(0))
            ).otherwise(0)
        ).alias("won_value"),

        sum(
            when(
                col("stage") == "lost",
                coalesce(col("deal_value"), lit(0))
            ).otherwise(0)
        ).alias("lost_value")
    )
)

# ------------------------------------------------------------
# 2. Touches → account grain
# ------------------------------------------------------------

touch_summary = (
    touches
    .groupBy("account_id")
    .agg(
        count("*").alias("touch_count")
    )
)

# ------------------------------------------------------------
# 3. Join account-level aggregates
# ------------------------------------------------------------

gold_account_sales = (
    accounts
    .select("account_id", "account_name")
    .dropDuplicates(["account_id"])
    .join(
        deal_summary,
        on="account_id",
        how="left"
    )
    .join(
        touch_summary,
        on="account_id",
        how="left"
    )
    .fillna({
        "total_deals": 0,
        "won_deals": 0,
        "lost_deals": 0,
        "won_value": 0,
        "lost_value": 0,
        "touch_count": 0
    })
)

# ------------------------------------------------------------
# 4. Write Gold Delta table
# ------------------------------------------------------------

(
    gold_account_sales
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_account_sales_summary")
)

print("✓ gold_account_sales_summary created")
print(f"Rows: {gold_account_sales.count()}")

display(
    gold_account_sales
    .orderBy(col("won_value").desc())
    .limit(20)
)

Deal stages:


stage
Lead
Proposal
Qualified
Negotiation
Lost
Won


✓ gold_account_sales_summary created
Rows: 100


account_id,account_name,total_deals,won_deals,lost_deals,won_value,lost_value,touch_count
100,Account 100,0,0,0,0.0,0.0,0
59,Account 59,0,0,0,0.0,0.0,0
63,Account 63,0,0,0,0.0,0.0,0
17,Account 17,5,0,0,0.0,0.0,0
76,Account 76,0,0,0,0.0,0.0,0
65,Account 65,0,0,0,0.0,0.0,0
11,Account 11,4,0,0,0.0,0.0,1
35,Account 35,0,0,0,0.0,0.0,0
32,Account 32,0,0,0,0.0,0.0,0
69,Account 69,0,0,0,0.0,0.0,0
